# LangSmith Datasets & Experiments

这个 Notebook 会从头构建本地 RAG，再使用官网手动上传的 `rag-evaluation-cases` Dataset 运行 Experiment。

流程：知识文件 → 切块 → Embedding → 向量库 → RAG → Dataset → Experiment。

## 1. 加载环境变量

必须先加载 `.env`，再导入 LangChain 和 LangSmith。

In [1]:
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(".env", usecwd=True)
if not env_path:
    raise FileNotFoundError("从当前工作目录向上没有找到 .env")

load_dotenv(env_path, override=True)

import os

required_env = [
    "LLM_MODEL",
    "LLM_BASE_URL",
    "LLM_API_KEY",
    "EMBEDDING_MODEL",
    "LANGSMITH_API_KEY",
]
missing = [name for name in required_env if not os.getenv(name)]
if missing:
    raise RuntimeError(f"缺少环境变量：{missing}")

project_root = Path(env_path).resolve().parent

print("项目目录：", project_root)
print("聊天模型：", os.getenv("LLM_MODEL"))
print("向量模型：", os.getenv("EMBEDDING_MODEL"))
print("LangSmith 项目：", os.getenv("LANGSMITH_PROJECT"))

项目目录： E:\AgentProject\studyAgent
聊天模型： qwen3.7-flash-2026-07-15
向量模型： qwen3.7-text-embedding-flash
LangSmith 项目： study


## 2. 加载 7 份知识文件

这些 Markdown 是 RAG 的知识库，不是 LangSmith Dataset。

In [2]:
from langchain_core.documents import Document

knowledge_dir = project_root / "03-evaluation" / "knowledge"
markdown_files = sorted(knowledge_dir.glob("*.md"))

if not markdown_files:
    raise FileNotFoundError(f"知识库中没有 Markdown 文件：{knowledge_dir}")

documents = [
    Document(
        page_content=file_path.read_text(encoding="utf-8"),
        metadata={"source": file_path.name},
    )
    for file_path in markdown_files
]

print(f"加载了 {len(documents)} 份知识文件：")
for document in documents:
    print("-", document.metadata["source"])

加载了 7 份知识文件：
- 01-leave-policy.md
- 02-expense-policy.md
- 03-remote-work-policy.md
- 04-attendance-policy.md
- 05-business-travel-policy.md
- 06-it-security-policy.md
- 07-learning-benefits.md


## 3. 文本切块

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    length_function=len,
    separators=["\n## ", "\n### ", "\n\n", "\n", "。", "！", "？", " ", ""],
)

chunks = text_splitter.split_documents(documents)
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = index

print(f"共生成 {len(chunks)} 个 chunk")
for chunk in chunks[:5]:
    print("\n---", chunk.metadata, "---")
    print(chunk.page_content)

共生成 8 个 chunk

--- {'source': '01-leave-policy.md', 'chunk_id': 0} ---
# 星河科技休假管理制度

## 适用范围

本制度适用于星河科技中国区全体正式员工。实习生和外包人员不享受本制度中的带薪年假。

## 带薪年假

正式员工每个自然年度享有 10 天带薪年假。入职不满一个自然年度的员工，年假按照当年度实际在职月份折算，计算结果不足半天的部分舍去。

年假最小申请单位为半天。连续申请 3 天及以上年假，需要至少提前 5 个工作日提交；申请少于 3 天，需要至少提前 2 个工作日提交。所有年假申请均由直属主管审批。

## 病假

员工申请 1 天以内的病假，可以先提交申请并在返岗后补充材料。连续病假达到 2 天或以上时，需要提供正规医疗机构出具的证明。

--- {'source': '01-leave-policy.md', 'chunk_id': 1} ---
## 年假结转

当年度未使用的年假最多可结转 5 天，结转部分必须在下一年度 3 月 31 日前使用，逾期自动失效。

--- {'source': '02-expense-policy.md', 'chunk_id': 2} ---
# 星河科技费用报销制度

## 提交时限

报销申请必须在费用发生后的 30 个自然日内提交。超过 30 个自然日的申请，需要部门负责人说明原因并额外审批。

## 办公费用审批

单笔金额低于 500 元的普通办公费用，由直属主管审批。单笔金额达到或超过 500 元但低于 5000 元时，需要直属主管和财务负责人共同审批。

单笔金额达到或超过 5000 元时，除直属主管和财务负责人外，还需要部门负责人审批。

## 报销凭证

所有报销必须提供合法有效的发票。电子发票需要上传原始 PDF 文件，截图不能代替原始电子发票。

## 不予报销的费用

个人娱乐、交通违章罚款以及未经批准购买的个人设备不属于公司报销范围。

--- {'source': '03-remote-work-policy.md', 'chunk_id': 3} ---
# 星河科技远程办公制度

## 可申请天数

通过试用期的员工每周最多可以申请 2 天远程办公。仍在试用期内的员工原则上应到办公室工作，特殊情况需要部门负

## 4. 创建 Embedding 和内存向量库

`check_embedding_ctx_length=False` 保证百炼接口收到原始字符串，而不是 token ID。

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(
    model=os.getenv("EMBEDDING_MODEL"),
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    check_embedding_ctx_length=False,
    encoding_format="float",
)

vector_store = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print(f"已将 {len(chunks)} 个 chunk 写入内存向量库")

E:\AgentProject\studyAgent\.venv\Lib\site-packages\langchain_openai\embeddings\base.py:359: UserWarning: WARNING! encoding_format is not default parameter.
                    encoding_format was transferred to model_kwargs.
                    Please confirm that encoding_format is what you intended.
  warnings.warn(


已将 8 个 chunk 写入内存向量库


## 5. 定义 Retriever 和 RAG

In [5]:
from langsmith import traceable
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


@traceable(run_type="retriever", name="retrieve_company_policies")
def retrieve(question: str, k: int = 3):
    return vector_store.similarity_search(question, k=k)


def format_context(retrieved_documents) -> str:
    sections = []
    for index, document in enumerate(retrieved_documents, start=1):
        source = document.metadata.get("source", "unknown")
        chunk_id = document.metadata.get("chunk_id", "unknown")
        sections.append(
            f"[资料 {index} | source={source} | chunk={chunk_id}]\n"
            f"{document.page_content}"
        )
    return "\n\n".join(sections)


llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_BASE_URL"),
    api_key=os.getenv("LLM_API_KEY"),
    temperature=0,
)

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """你是星河科技的公司制度助手。
只能依据下面的知识库上下文回答，不要使用自己的常识补充公司制度。
如果上下文不足以回答，必须回答“根据当前知识库无法确定”。
回答应简洁、准确。

知识库上下文：
{context}""",
    ),
    ("human", "{question}"),
])


@traceable(name="company_policy_rag")
def ask_rag(question: str) -> dict:
    retrieved_documents = retrieve(question, k=3)
    context = format_context(retrieved_documents)
    messages = rag_prompt.invoke({"question": question, "context": context})
    response = llm.invoke(messages)

    return {
        "answer": response.content,
        "context": context,
        "sources": [
            document.metadata.get("source", "unknown")
            for document in retrieved_documents
        ],
    }

## 6. 先手动测试一次 RAG

这一步成功后再运行 Experiment，否则 28 道题会一起失败。

In [6]:
test_result = ask_rag("800元办公费用需要谁审批？")

print("回答：", test_result["answer"])
print("来源：", test_result["sources"])
print("\n检索上下文：\n", test_result["context"])

回答： 根据《星河科技费用报销制度》，单笔金额达到或超过500元但低于5000元的办公费用需由直属主管和财务负责人共同审批。因此，800元办公费用需要**直属主管和财务负责人**共同审批。
来源： ['02-expense-policy.md', '03-remote-work-policy.md', '07-learning-benefits.md']

检索上下文：
 [资料 1 | source=02-expense-policy.md | chunk=2]
# 星河科技费用报销制度

## 提交时限

报销申请必须在费用发生后的 30 个自然日内提交。超过 30 个自然日的申请，需要部门负责人说明原因并额外审批。

## 办公费用审批

单笔金额低于 500 元的普通办公费用，由直属主管审批。单笔金额达到或超过 500 元但低于 5000 元时，需要直属主管和财务负责人共同审批。

单笔金额达到或超过 5000 元时，除直属主管和财务负责人外，还需要部门负责人审批。

## 报销凭证

所有报销必须提供合法有效的发票。电子发票需要上传原始 PDF 文件，截图不能代替原始电子发票。

## 不予报销的费用

个人娱乐、交通违章罚款以及未经批准购买的个人设备不属于公司报销范围。

[资料 2 | source=03-remote-work-policy.md | chunk=3]
# 星河科技远程办公制度

## 可申请天数

通过试用期的员工每周最多可以申请 2 天远程办公。仍在试用期内的员工原则上应到办公室工作，特殊情况需要部门负责人批准。

## 申请流程

远程办公申请需要至少提前 1 个工作日在内部系统提交，并获得直属主管批准。未经批准自行远程办公按缺勤处理。

## 工作要求

远程办公期间，员工应在工作时间保持即时通信工具在线，并参加安排的线上会议。涉及客户纸质资料或公司保密设备的工作不得带离办公室。

## 不适用日期

公司级培训日、季度复盘日以及明确要求线下参加的客户会议日，不得安排远程办公。

[资料 3 | source=07-learning-benefits.md | chunk=7]
# 星河科技学习与认证福利

## 年度学习额度

通过试用期的正式员工每个自然年度可以申请最高 3000 元学习费用。额度不能跨年度结转，也不能转让

## 7. 定义 Target 和 Evaluator

请先在官网手动上传 CSV，并确认 Dataset 名称是 `rag-evaluation-cases`，字段为 `Input.question` 和 `Reference Output.reference_answer`。

In [7]:
dataset_name = "rag-evaluation-cases"


def rag_target(inputs: dict) -> dict:
    """LangSmith 会把每条 Dataset Input 传入这里。"""
    result = ask_rag(inputs["question"])
    return {
        "answer": result["answer"],
        "retrieved_sources": result["sources"],
        "retrieved_context": result["context"],
    }


def exact_match(outputs: dict, reference_outputs: dict) -> bool:
    """最小示例：实际答案必须与 CSV 参考答案逐字一致。"""
    actual = outputs["answer"].strip()
    expected = reference_outputs["reference_answer"].strip()
    return actual == expected


from pydantic import BaseModel, Field


class SemanticGrade(BaseModel):
    """语义裁判的结构化输出。"""

    correct: bool = Field(description="实际答案与参考答案的含义是否一致")
    reason: str = Field(description="简短说明判断依据")


semantic_judge = llm.with_structured_output(SemanticGrade)


def semantic_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    """使用 LLM 判断含义是否正确，不要求文字完全一致。"""
    grade = semantic_judge.invoke([
        (
            "system",
            "你是严格的答案评估员。判断实际答案是否表达了参考答案的核心事实。"
            "允许措辞、语序和详略不同；如果遗漏关键条件、包含矛盾或无依据扩展，则判为错误。",
        ),
        (
            "human",
            f"问题：{inputs['question']}\n"
            f"参考答案：{reference_outputs['reference_answer']}\n"
            f"实际答案：{outputs['answer']}",
        ),
    ])
    return {
        "key": "semantic_correctness",
        "score": 1 if grade.correct else 0,
        "comment": grade.reason,
    }

## 8. 运行 Experiment

这会真实运行 Dataset 中的全部问题并产生模型调用费用。`max_concurrency=1` 表示逐题执行。

In [8]:
from langsmith import evaluate

experiment_results = evaluate(
    rag_target,
    data=dataset_name,
    evaluators=[exact_match, semantic_correctness],
    experiment_prefix="rag-semantic-evaluation experiment",
    max_concurrency=1,
)

print(experiment_results)

View the evaluation results for experiment: 'rag-semantic-evaluation experiment-dde93232' at:
https://smith.langchain.com/o/0186ad00-74ec-4262-bd54-e3ce799d852c/datasets/7583cba1-aa6f-42c4-866e-36aff8391764/compare?selectedSessions=6ef9be42-cff6-4c48-9c1d-0f38545685ce




Error running evaluator <DynamicRunEvaluator semantic_correctness> on run 01a0747b-a530-7570-b9db-bb04a3b61ce6: OpenAIPermissionDeniedError('Error code: 403 - {\'error\': {\'message\': \'Free quota exhausted. To continue accessing the model on a paid basis, please add funds or disable the "use free tier only" mode in the management console.\', \'type\': \'AllocationQuota.FreeTierOnly\', \'param\': None, \'code\': \'AllocationQuota.FreeTierOnly\'}, \'id\': \'chatcmpl-0f1bed83-5830-9bf4-ac64-b38531fa553a\', \'request_id\': \'0f1bed83-5830-9bf4-ac64-b38531fa553a\'}')
Traceback (most recent call last):
  File "E:\AgentProject\studyAgent\.venv\Lib\site-packages\langchain_openai\chat_models\base.py", line 1843, in _generate
    self.root_client.chat.completions.with_raw_response.parse(**payload)
  File "E:\AgentProject\studyAgent\.venv\Lib\site-packages\openai\_legacy_response.py", line 369, in wrapped
    return cast(LegacyAPIResponse[R], func(*args, **kwargs))
                             

<ExperimentResults rag-semantic-evaluation experiment-dde93232>


## 9. 查看结果

结果中会同时出现两个指标：`exact_match` 表示文字是否逐字一致；`semantic_correctness` 表示答案含义是否正确。判断问答质量时主要看后者，前者只作为格式稳定性的参考。

语义评分会为每道题额外调用一次 Judge 模型，因此评测调用量大约是原来的两倍。

位置：`Datasets & Experiments → rag-evaluation-cases → Experiments`。